# 07 — Trading Parameter Grid Search

Loads the **pre-trained LightGBM model** and **selected features** from `models/`,
runs a fast backtest across a grid of trading parameters, and ranks every combination
by a chosen optimisation metric.

**What is searched (trading params only — model is frozen):**

| Parameter | Values |
|---|---|
| `long_threshold` | entry confidence floor for longs |
| `short_threshold` | entry confidence ceiling for shorts |
| `atr_multiplier` | SL width in ATR units |
| `min_sl` | SL floor (%) |
| `take_profit` | TP target (%) |
| `min_hold` | minimum candles before conf-exit |
| `max_hold` | force-exit candles |
| `cooldown` | lockout candles after exit |

**Optimisation metric (configurable):** Sharpe ratio on the test set by default.
Alternatives: total return, Calmar ratio, win rate, profit factor.

## 1. Config

In [ ]:
SYMBOL   = 'BTCUSDT'
INTERVAL = '1h'

TRAIN_END = '2024-06-01'
VAL_END   = '2024-11-10'
# Test: VAL_END → present (same split as 06)

# ── Optimisation ──────────────────────────────────────────────────────────────
# Metric used to rank grid results.
# Options: 'sharpe', 'total_return', 'calmar', 'win_rate', 'profit_factor'
OPTIMISE_METRIC = 'sharpe'

# Minimum trades required for a result to be considered valid.
# Filters out degenerate configs with 1-2 lucky trades.
MIN_TRADES = 30

# Top N results to display in the leaderboard
TOP_N = 20

# ── Search grid ───────────────────────────────────────────────────────────────
# Each list is a set of candidate values. Total combinations = product of all lengths.
# Current grid: ~5k combinations, typically runs in 2-4 min.
GRID = {
    'long_threshold':  [0.54, 0.55, 0.57, 0.59, 0.61],
    'short_threshold': [0.39, 0.41, 0.43, 0.45, 0.46],
    'atr_multiplier':  [1.5, 2.0, 2.5],
    'min_sl':          [0.010, 0.015, 0.020],
    'take_profit':     [0.025, 0.030, 0.040],
    'min_hold':        [4, 6, 8],
    'max_hold':        [24, 48],
    'cooldown':        [2, 3],
}
# Constraint: long_threshold + short_threshold must leave a dead-band.
# Enforced at search time: long_threshold > (1 - short_threshold) + 0.05


## 2. Imports & paths

In [ ]:
import itertools
import json
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

from hmats.data.splits import calendar_split

warnings.filterwarnings('ignore')

mpl.rcParams.update({
    'font.family': 'serif', 'font.serif': ['DejaVu Serif'],
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.labelsize': 10, 'axes.titlesize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 9, 'legend.framealpha': 0.85,
    'figure.dpi': 120, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})
ACCENT='#F7931A'; BLUE='#2962FF'; GREY='#9E9E9E'
RED='#EF5350'; GREEN='#26A69A'; PURPLE='#7B1FA2'

REPO_ROOT = Path.cwd().parents[2]
if not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = Path.cwd()

FEATURES_DIR = REPO_ROOT / 'data' / 'features'
MODELS_DIR   = REPO_ROOT / 'models'
FIGURES_DIR  = REPO_ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

## 3. Load model & pre-computed predictions

In [ ]:
# Load pre-trained model from 06
model = lgb.Booster(model_file=str(MODELS_DIR / 'lgbm_model.txt'))
selected_features = pd.read_csv(
    MODELS_DIR / 'lgbm_features.csv', header=None
)[0].tolist()

print(f'Model loaded  — {model.num_trees()} trees')
print(f'Features used — {len(selected_features)}')
print(selected_features[:10], '...')


In [ ]:
# Load feature parquet and split — must match 06 exactly
feat_df = pd.read_parquet(FEATURES_DIR / f'{SYMBOL}_{INTERVAL}_features.parquet')
feat_df.index = feat_df.index.tz_localize(None) if feat_df.index.tz else feat_df.index

BACKTEST_COLS = ['close', 'sma_200', 'atr_14_pct']

_, val_df, test_df = calendar_split(feat_df, train_end=TRAIN_END, val_end=VAL_END)

# Predictions — computed once, reused across all grid iterations
X_val  = val_df[selected_features].values
X_test = test_df[selected_features].values

probs_val  = model.predict(X_val)
probs_test = model.predict(X_test)

close_arr  = test_df['close'].values
atr_arr    = test_df['atr_14_pct'].values
sig_index  = test_df.index

bh_units  = 1.0 / close_arr[0]
bh_equity = bh_units * close_arr
bh_final  = bh_equity[-1]

print(f'Val  rows: {len(val_df):,}   {val_df.index.min().date()} → {val_df.index.max().date()}')
print(f'Test rows: {len(test_df):,}  {test_df.index.min().date()} → {test_df.index.max().date()}')
print(f'Prob range: [{probs_test.min():.3f}, {probs_test.max():.3f}]')


## 4. Backtest engine & scoring

In [ ]:
def run_backtest(probs, close_arr, atr_arr, sig_index, params):
    """
    Single-pass vectorised-style backtest.
    Returns (equity_arr, trades_df).
    All params come from the params dict.
    """
    long_thr  = params['long_threshold']
    short_thr = params['short_threshold']
    exit_long  = 1 - long_thr   # symmetric exit
    exit_short = 1 - short_thr
    atr_mult  = params['atr_multiplier']
    min_sl    = params['min_sl']
    tp        = params['take_profit']
    min_hold  = params['min_hold']
    max_hold  = params['max_hold']
    cooldown_n= params['cooldown']

    cash = 1.0; units = 0.0
    in_pos = False; direction = None
    entry_px = 0.0; entry_ts = None; entry_cash = 0.0
    dynamic_sl = 0.0; hold_count = 0; cooldown = 0

    equity_curve = [1.0]
    trade_log    = []

    for ts, px, conf, atr_pct in zip(sig_index, close_arr, probs, atr_arr):
        if cooldown > 0:
            cooldown -= 1

        if in_pos:
            hold_count += 1
            pnl = (px - entry_px) / entry_px if direction == 'long'                   else (entry_px - px) / entry_px
            reason = None
            if   pnl <= -dynamic_sl:          reason = 'sl'
            elif pnl >= tp:                   reason = 'tp'
            elif hold_count >= max_hold:       reason = 'max_hold'
            elif hold_count >= min_hold:
                if direction == 'long'  and conf < exit_long:  reason = 'conf'
                elif direction == 'short' and conf > exit_short: reason = 'conf'

            if reason:
                cash  = units * px if direction == 'long' else entry_cash * (1 + pnl)
                units = 0.0
                trade_log.append({'direction': direction, 'pnl_pct': pnl,
                                   'hold_h': hold_count, 'reason': reason})
                in_pos = False; direction = None
                hold_count = 0; cooldown = cooldown_n

        if not in_pos and cooldown == 0:
            sl = max(atr_mult * atr_pct, min_sl)
            if conf >= long_thr:
                units = cash / px; cash = 0.0
                in_pos = True; direction = 'long'
                entry_px = px; entry_ts = ts; hold_count = 0; dynamic_sl = sl
            elif conf <= short_thr:
                entry_cash = cash; units = cash / px
                in_pos = True; direction = 'short'
                entry_px = px; entry_ts = ts; hold_count = 0; dynamic_sl = sl

        if in_pos and direction == 'long':
            equity_curve.append(units * px)
        elif in_pos and direction == 'short':
            equity_curve.append(entry_cash * (1 + (entry_px - px) / entry_px))
        else:
            equity_curve.append(cash)

    # Force close
    if in_pos:
        px  = close_arr[-1]
        pnl = (px - entry_px)/entry_px if direction == 'long' else (entry_px - px)/entry_px
        cash = units * px if direction == 'long' else entry_cash * (1 + pnl)
        trade_log.append({'direction': direction, 'pnl_pct': pnl,
                           'hold_h': hold_count, 'reason': 'eod'})
        equity_curve[-1] = cash

    return np.array(equity_curve[1:]), pd.DataFrame(trade_log)


def score(equity_arr, trades_df, metric, bh_final):
    """Compute the optimisation metric for one backtest run."""
    if trades_df.empty:
        return -np.inf

    eq = equity_arr
    ret = np.log(eq[1:] / (eq[:-1] + 1e-12))
    ann = 24 * 365

    if metric == 'sharpe':
        return float(ret.mean() / (ret.std(ddof=1) + 1e-12) * np.sqrt(ann))

    elif metric == 'total_return':
        return float(eq[-1] - 1)

    elif metric == 'calmar':
        ann_ret = float((eq[-1] ** (ann / len(eq))) - 1)
        pk = np.maximum.accumulate(eq)
        mdd = float(((eq - pk) / (pk + 1e-12)).min())
        return ann_ret / (abs(mdd) + 1e-6)

    elif metric == 'win_rate':
        return float((trades_df['pnl_pct'] > 0).mean())

    elif metric == 'profit_factor':
        gains  = trades_df[trades_df['pnl_pct'] > 0]['pnl_pct'].sum()
        losses = trades_df[trades_df['pnl_pct'] < 0]['pnl_pct'].abs().sum()
        return float(gains / (losses + 1e-6))

    return -np.inf


## 5. Grid search

In [ ]:
# Build all valid param combinations
keys   = list(GRID.keys())
combos = list(itertools.product(*[GRID[k] for k in keys]))

# Apply constraint: long_threshold must be meaningfully above 0.5,
# short_threshold below 0.5, and the two thresholds must not overlap.
valid_combos = []
for vals in combos:
    p = dict(zip(keys, vals))
    # dead-band: long_thr and (1 - short_thr) must differ by at least 0.02
    if p['long_threshold'] - (1 - p['short_threshold']) < 0.02:
        continue
    # symmetric: short_threshold <= 0.5 and long_threshold >= 0.5
    if p['short_threshold'] >= 0.5 or p['long_threshold'] <= 0.5:
        continue
    valid_combos.append(p)

print(f'Total combinations : {len(combos):,}')
print(f'Valid after filter : {len(valid_combos):,}')
print(f'Optimising for     : {OPTIMISE_METRIC}')
print(f'Minimum trades     : {MIN_TRADES}')


In [ ]:
results = []

for params in tqdm(valid_combos, desc='Grid search'):
    eq, tdf = run_backtest(probs_test, close_arr, atr_arr, sig_index, params)

    n_trades = len(tdf)
    if n_trades < MIN_TRADES:
        continue

    s = score(eq, tdf, OPTIMISE_METRIC, bh_final)

    pk   = np.maximum.accumulate(eq)
    mdd  = float(((eq - pk) / (pk + 1e-12)).min())
    ret  = np.log(eq[1:] / (eq[:-1] + 1e-12))
    shrp = float(ret.mean() / (ret.std(ddof=1) + 1e-12) * np.sqrt(24*365))

    results.append({
        **params,
        'score':        s,
        'total_return': float(eq[-1] - 1),
        'sharpe':       shrp,
        'max_dd':       mdd,
        'n_trades':     n_trades,
        'win_rate':     float((tdf['pnl_pct'] > 0).mean()),
        'n_long':       int((tdf['direction'] == 'long').sum()),
        'n_short':      int((tdf['direction'] == 'short').sum()),
        'avg_hold':     float(tdf['hold_h'].mean()),
        'n_sl':         int((tdf['reason'] == 'sl').sum()),
        'n_tp':         int((tdf['reason'] == 'tp').sum()),
        'n_conf':       int((tdf['reason'] == 'conf').sum()),
    })

results_df = pd.DataFrame(results).sort_values('score', ascending=False).reset_index(drop=True)
print(f'\nValid results : {len(results_df):,}')
print(f'Best {OPTIMISE_METRIC}    : {results_df["score"].iloc[0]:.4f}')
print(f'Worst {OPTIMISE_METRIC}   : {results_df["score"].iloc[-1]:.4f}')


## 6. Leaderboard

In [ ]:
display_cols = [
    'score', 'total_return', 'sharpe', 'max_dd', 'win_rate',
    'n_trades', 'n_long', 'n_short', 'n_sl', 'n_tp',
    'long_threshold', 'short_threshold',
    'atr_multiplier', 'min_sl', 'take_profit',
    'min_hold', 'max_hold', 'cooldown',
]

top = results_df[display_cols].head(TOP_N).copy()
top['total_return'] = top['total_return'].map('{:+.2%}'.format)
top['max_dd']       = top['max_dd'].map('{:.2%}'.format)
top['win_rate']     = top['win_rate'].map('{:.1%}'.format)
top['score']        = top['score'].map('{:.4f}'.format)
top['sharpe']       = top['sharpe'].map('{:.3f}'.format)

print(f'Top {TOP_N} by {OPTIMISE_METRIC}:\n')
print(top.to_string(index=True))

best = results_df.iloc[0]
print(f'\n── Best config ──────────────────────────────────────────')
for k in keys:
    print(f'  {k:<20}: {best[k]}')
print(f'  {"score (" + OPTIMISE_METRIC + ")":<20}: {best["score"]:.4f}')
print(f'  {"total_return":<20}: {best["total_return"]:+.2%}')
print(f'  {"sharpe":<20}: {best["sharpe"]:.3f}')
print(f'  {"max_dd":<20}: {best["max_dd"]:.2%}')
print(f'  {"n_trades":<20}: {int(best["n_trades"])}')


## 7. Result distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

plot_cols = [
    ('score',        f'Optimisation metric ({OPTIMISE_METRIC})', BLUE),
    ('total_return', 'Total return',                              ACCENT),
    ('sharpe',       'Sharpe (ann.)',                             BLUE),
    ('max_dd',       'Max drawdown',                             RED),
    ('win_rate',     'Win rate',                                  GREEN),
    ('n_trades',     'Number of trades',                         GREY),
]

for ax, (col, title, color) in zip(axes, plot_cols):
    data = results_df[col].dropna()
    ax.hist(data, bins=50, color=color, alpha=0.75, edgecolor='none')
    ax.axvline(data.median(), color='black', lw=1.2, ls='--', label=f'Median {data.median():.3f}')
    ax.axvline(results_df[col].iloc[0], color=ACCENT, lw=1.5, ls='-', label=f'Best {results_df[col].iloc[0]:.3f}')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(col)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle(f'Grid search result distributions  (n={len(results_df):,} valid configs)',
             fontweight='bold', fontsize=12)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'grid_search_distributions.png')
plt.show()


## 8. Parameter sensitivity

In [ ]:
# For each param, show how the median score varies across its values
# — tells you which parameters matter most.
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for ax, param in zip(axes, keys):
    grp = results_df.groupby(param)['score'].agg(['median', 'mean', 'std']).reset_index()
    x   = grp[param].astype(str)
    ax.bar(x, grp['median'], color=BLUE, alpha=0.7, label='Median')
    ax.errorbar(x, grp['median'], yerr=grp['std'], fmt='none',
                color='black', capsize=4, lw=1.2)
    ax.set_title(param, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel(f'Median {OPTIMISE_METRIC}')
    ax.grid(axis='y', alpha=0.3)
    # Highlight best value
    best_val = str(best[param])
    for tick in ax.get_xticklabels():
        if tick.get_text() == best_val:
            tick.set_color(ACCENT)
            tick.set_fontweight('bold')

# Hide unused axes
for ax in axes[len(keys):]:
    ax.set_visible(False)

fig.suptitle(f'Parameter sensitivity — median {OPTIMISE_METRIC} by param value\n'
             f'(orange x-labels = value chosen by best config)',
             fontweight='bold', fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'grid_search_sensitivity.png')
plt.show()


## 9. Best config — full backtest & equity curve

In [ ]:
best_params = {k: best[k] for k in keys}
eq_best, tdf_best = run_backtest(probs_test, close_arr, atr_arr, sig_index, best_params)

# ── Equity curve ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 8),
                          gridspec_kw={'height_ratios': [3, 1.2], 'hspace': 0.08})

ax = axes[0]
ax.plot(sig_index, eq_best,   color=ACCENT, lw=1.4, label='Best grid config')
ax.plot(sig_index, bh_equity, color=BLUE,   lw=1.2, ls='--', label='Buy & Hold')
ax.axhline(1.0, color=GREY, lw=0.7, ls=':')
ax.set_ylabel('Portfolio value (start=1.0)')
ax.set_title('Best grid config — equity curve vs Buy & Hold', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3); ax.grid(axis='x', alpha=0.15)

ax = axes[1]
pk_b = np.maximum.accumulate(eq_best)
pk_h = np.maximum.accumulate(bh_equity)
ax.fill_between(sig_index, (eq_best - pk_b)/(pk_b + 1e-12)*100, 0,
                color=ACCENT, alpha=0.45, label='Best config')
ax.fill_between(sig_index, (bh_equity - pk_h)/(pk_h + 1e-12)*100, 0,
                color=BLUE,   alpha=0.25, label='Buy & Hold')
ax.set_ylabel('Drawdown (%)'); ax.set_title('Underwater equity')
ax.legend(); ax.grid(axis='y', alpha=0.3); ax.grid(axis='x', alpha=0.15)

for ax in axes:
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'grid_best_equity.png')
plt.show()

In [ ]:
# Summary of best config
pnl_b = tdf_best['pnl_pct']
print('── Best config summary ──────────────────────────────────')
for k, v in best_params.items():
    print(f'  {k:<22}: {v}')
print()
print(f'  {"Total return":<22}: {(eq_best[-1]-1)*100:+.2f}%')
print(f'  {"Buy & Hold return":<22}: {(bh_final-1)*100:+.2f}%')
print(f'  {"Alpha":<22}: {(eq_best[-1] - bh_final)*100:+.2f}pp')
print(f'  {"Sharpe (ann.)":<22}: {best["sharpe"]:.3f}')
print(f'  {"Max drawdown":<22}: {best["max_dd"]*100:.2f}%')
print(f'  {"Calmar":<22}: {best["score"] if OPTIMISE_METRIC=="calmar" else "—"}')
print()
print(f'  {"Total trades":<22}: {len(tdf_best)}')
print(f'    {"Long":<20}: {(tdf_best["direction"]=="long").sum()}')
print(f'    {"Short":<20}: {(tdf_best["direction"]=="short").sum()}')
print(f'  {"Win rate":<22}: {(pnl_b > 0).mean()*100:.1f}%')
print()
print(f'  Return percentiles:')
for p in [5, 25, 50, 75, 95]:
    print(f'    p{p:<3}: {pnl_b.quantile(p/100)*100:+.2f}%')


## 10. Save results

In [ ]:
RESULTS_PATH = REPO_ROOT / 'models' / 'grid_search_results.csv'
results_df.to_csv(RESULTS_PATH, index=False)
print(f'Saved {len(results_df):,} results → {RESULTS_PATH}')

# Also save best params as JSON for easy loading in 06 or production
BEST_PARAMS_PATH = REPO_ROOT / 'models' / 'best_trading_params.json'
with open(BEST_PARAMS_PATH, 'w') as f:
    json.dump({
        'params':   best_params,
        'metric':   OPTIMISE_METRIC,
        'score':    float(best['score']),
        'sharpe':   float(best['sharpe']),
        'total_return': float(best['total_return']),
        'max_dd':   float(best['max_dd']),
        'n_trades': int(best['n_trades']),
    }, f, indent=2)
print(f'Saved best params  → {BEST_PARAMS_PATH}')
